# SQL for Data Analysis | Seminar 2

Today we will learn how you may connect to your database via  Python script and use this data for your advantage. Today's topic will be devoted to the following topics:

- SELECT and FROM
- Filtering via WHERE
- JOINS

If all goes well enough, we might also cover such topics as aggregate functions and GROUP BY statement

## Setup

Install dependencies (if necessary):

In [ ]:
# !pip install sqlalchemy psycopg2-binary ipython-sql pandas

## Connection

In [1]:
from sqlalchemy import create_engine, text
import pandas as pd

# ============================================================
# Our credentials
# ============================================================
DB_USER     = ""
DB_PASSWORD = ""
DB_HOST     = ""
DB_PORT     = ""
DB_NAME     = ""
# ============================================================

CONNECTION_STRING = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(CONNECTION_STRING)

def sql(query: str):
    """Execute a SQL query and return results as a DataFrame."""
    with engine.connect() as conn:
        return pd.read_sql(text(query), conn)


result = sql('select 1 as one')



In [3]:
result = sql('select 1 as one')
result

,one
0,1


### Helper function

Basically *sqlalchemy* allows us to create engines of our own to connect to anything via python scripts

Run any SQL and get a nice DataFrame back:

In [4]:

sql("""
    SELECT table_name, table_type
    FROM information_schema.tables
    WHERE table_schema = 'public'
    ORDER BY table_type, table_name
""")

,table_name,table_type
0,actor,BASE TABLE
1,actors_full,BASE TABLE
2,address,BASE TABLE
3,category,BASE TABLE
4,city,BASE TABLE
5,country,BASE TABLE
6,customer,BASE TABLE
7,film,BASE TABLE
8,film_actor,BASE TABLE
9,film_category,BASE TABLE


---
## SELECT, FROM and JOINS



### SELECT — basic queries

The query below:
- Returns columns such as *title*, *description*, *release_year* from table *public.films*
- Filters them by *release_year* and *language_id*

As a result you get a table with Italian films from 2016

In [5]:

sql("""
select
	title,
	description,
	release_year
from film
where 1=1
	and release_year = 2016
	and language_id = 2
  """)








,title,description,release_year
0,ANNIE IDENTITY,A Amazing Panorama of a Pastry Chef And a Boat...,2016
1,BREAKING HOME,A Beautiful Display of a Secret Agent And a Mo...,2016
2,CENTER DINOSAUR,A Beautiful Character Study of a Sumo Wrestler...,2016
3,DRUMLINE CYCLONE,A Insightful Panorama of a Monkey And a Sumo W...,2016
4,HEAVENLY GUN,A Beautiful Yarn of a Forensic Psychologist An...,2016
5,REEF SALUTE,A Action-Packed Saga of a Teacher And a Lumber...,2016


This query does exactly the same thing as the one from before, but this time we are user ORDER BY command that allows us to set the order in which we want to return the data

In [7]:
# SELECT statement with ordering


sql("""
select
	title,
	description,
	release_year,
	rental_rate
from film
where 1=1
	and release_year = 2016
	and language_id = 2
order by rental_rate

""")

,title,description
0,ANNIE IDENTITY,A Amazing Panorama of a Pastry Chef And a Boat...
1,DRUMLINE CYCLONE,A Insightful Panorama of a Monkey And a Sumo W...
2,REEF SALUTE,A Action-Packed Saga of a Teacher And a Lumber...
3,BREAKING HOME,A Beautiful Display of a Secret Agent And a Mo...
4,CENTER DINOSAUR,A Beautiful Character Study of a Sumo Wrestler...
5,HEAVENLY GUN,A Beautiful Yarn of a Forensic Psychologist An...


In [8]:
#SELECT statement with ordering and OR statement
sql("""
select
	title,
	description,
	release_year,
	rental_rate,
	length
from film
where 1=1
	and release_year = 2016
	and language_id = 2
	or  length <= 60
order by rental_rate desc

""")

,title,description,release_year,rental_rate,length
0,MATRIX SNOWMAN,A Action-Packed Saga of a Womanizer And a Woma...,2010,4.99,56
1,DAWN POND,A Thoughtful Documentary of a Dentist And a Fo...,2007,4.99,57
2,DEEP CRUSADE,A Amazing Tale of a Crocodile And a Squirrel w...,2024,4.99,51
3,DESTINY SATURDAY,A Touching Drama of a Crocodile And a Crocodil...,2022,4.99,56
4,MOVIE SHAKESPEARE,A Insightful Display of a Database Administrat...,2017,4.99,53
...,...,...,...,...,...
104,SUNSET RACER,A Awe-Inspiring Reflection of a Astronaut And ...,2007,0.99,48
105,VALENTINE VANISHING,A Thrilling Display of a Husband And a Butler ...,2006,0.99,48
106,VISION TORQUE,A Thoughtful Documentary of a Dog And a Man wh...,2020,0.99,59
107,WESTWARD SEABISCUIT,A Lacklusture Tale of a Butler And a Husband w...,2012,0.99,52


In [9]:
sql("""
select
	title,
	language_id,
	description,
	release_year,
	rental_rate,
	length,
	rental_duration*rental_rate as total_rate
from film
where 1=1
	and release_year = 2016
	and language_id = 2
	or  (length <= 60
		and rental_duration*rental_rate <= 30)
order by rental_rate desc

""")

,title,language_id,description,release_year,rental_rate,length,total_rate
0,GOODFELLAS SALUTE,3,A Unbelieveable Tale of a Dog And a Explorer w...,2021,4.99,56,19.96
1,GUMP DATE,1,A Intrepid Yarn of a Explorer And a Student wh...,2023,4.99,53,14.97
2,HALL CASSIDY,5,A Beautiful Panorama of a Pastry Chef And a A ...,2019,4.99,51,24.95
3,ACE GOLDFINGER,1,A Astounding Epistle of a Database Administrat...,2023,4.99,48,14.97
4,HANOVER GALAXY,4,A Stunning Reflection of a Girl And a Secret A...,2016,4.99,47,24.95
...,...,...,...,...,...,...,...
101,LEGEND JEDI,4,A Awe-Inspiring Epistle of a Pioneer And a Stu...,2016,0.99,59,6.93
102,LION UNCUT,1,A Intrepid Display of a Pastry Chef And a Cat ...,2020,0.99,50,5.94
103,GO PURPLE,6,A Fast-Paced Display of a Car And a Database A...,2015,0.99,54,2.97
104,CABIN FLASH,1,A Stunning Epistle of a Boat And a Man who mus...,2006,0.99,53,3.96


In [ ]:
# INNER JOIN - 2 types to join tables

sql("""


select
	film_category.category_id
FROM film_category
JOIN category on film_category.category_id  = category.category_id
""")


sql("""
select
	category_id
FROM film_category
JOIN category USING(category_id)

""")

In [ ]:
# INNER JOIN - Filtering data

sql(""""

select
	title,
	language_id,
	description,
	release_year,
	rental_rate,
	length,
	rental_duration*rental_rate as total_rate,
	category."name" category_name
FROM film_category
JOIN category USING(category_id)
JOIN film USING(film_id)
where 1=1
	and category."name" = 'Horror'
	and release_year < 2010

""")

In [ ]:
# Film count per category

#COUNT, MAX, MIN, AVG, SUM

sql("""
    SELECT
      name,
      count(distinct film_id),
      min(rental_rate),
      max(rental_rate),
      sum(rental_rate),
      avg(rental_rate)
    FROM film_category
    JOIN category USING(category_id)
    JOIN film USING(film_id)
    GROUP BY name
    """)




,name,count,min,max,sum,avg
0,Action,149,0.99,4.99,467.51,3.137651
1,Animation,148,0.99,4.99,474.52,3.206216
2,Children,150,0.99,4.99,456.50,3.043333
3,Classics,147,0.99,4.99,427.53,2.908367
4,Comedy,143,0.99,4.99,421.57,2.948042
5,Documentary,145,0.99,4.99,433.55,2.990000
6,Drama,152,0.99,4.99,432.48,2.845263
7,Family,147,0.99,4.99,411.53,2.799524
8,Foreign,150,0.99,4.99,450.50,3.003333
9,Games,150,0.99,4.99,440.50,2.936667


### SELECT — JOINs

In [ ]:
# Film count per category

#COUNT, MAX, MIN, AVG, SUM

sql("""


    SELECT
      CONCAT(actor.first_name, ' ', actor.last_name),
      count(distinct film.film_id) as films_amount
    FROM film_category
      JOIN category USING(category_id)
      JOIN film USING(film_id)
      JOIN language USING(language_id)
      JOIN film_actor USING(film_id)
      JOIN actor USING(actor_id)
    WHERE category.name = 'Horror'
    GROUP BY CONCAT(actor.first_name, ' ', actor.last_name)

    """)




,concat,films_amount
0,ADAM GRANT,1
1,ADAM HOPPER,4
2,ALAN DREYFUSS,3
3,ALBERT JOHANSSON,5
4,ALBERT NOLTE,6
...,...,...
190,WILLIAM HACKMAN,3
191,WILL WILSON,6
192,WOODY HOFFMAN,8
193,WOODY JOLIE,7


In [ ]:
sql("""


with actors_horror as (

    SELECT
      CONCAT(actor.first_name, ' ', actor.last_name) as actor_name,
      count(distinct film.film_id) as films_amount
    FROM film_category
      JOIN category USING(category_id)
      JOIN film USING(film_id)
      JOIN language USING(language_id)
      JOIN film_actor USING(film_id)
      JOIN actor USING(actor_id)
    WHERE category.name = 'Horror'
    GROUP BY CONCAT(actor.first_name, ' ', actor.last_name)

  )

  select
    actor_name,
    films_amount
  from actors_horror
  order by films_amount

    """)




sql("""

drop table if exists actors_horror;
create temporary table actors_horror as (

    SELECT
      CONCAT(actor.first_name, ' ', actor.last_name) as actor_name,
      count(distinct film.film_id) as films_amount
    FROM film_category
      JOIN category USING(category_id)
      JOIN film USING(film_id)
      JOIN language USING(language_id)
      JOIN film_actor USING(film_id)
      JOIN actor USING(actor_id)
    WHERE category.name = 'Horror'
    GROUP BY CONCAT(actor.first_name, ' ', actor.last_name)

  );

  select
    actor_name,
    films_amount
  from actors_horror
  order by films_amount

    """)

,actor_name,films_amount
0,MICHAEL BENING,1
1,HARRISON BALE,1
2,FAY WOOD,1
3,BURT TEMPLE,1
4,TOM MCKELLEN,1
...,...,...
190,JUDE CRUISE,8
191,RIP CRAWFORD,9
192,GROUCHO DUNST,10
193,WARREN NOLTE,10


### SELECT — Aggregation

In [ ]:
# Top 10 customers by total spending
sql("""
    SELECT c.customer_id,
           CONCAT(c.first_name, ' ', c.last_name) AS customer,
           SUM(p.amount) AS total_spent
    FROM customer c
    JOIN payment p USING (customer_id)
    GROUP BY c.customer_id, c.first_name, c.last_name
    ORDER BY total_spent DESC
    LIMIT 10
""")

In [ ]:
# Revenue per store
sql("""
    SELECT s.store_id,
           COUNT(p.payment_id) AS num_payments,
           SUM(p.amount) AS total_revenue
    FROM payment p
    JOIN staff st USING (staff_id)
    JOIN store s ON st.store_id = s.store_id
    GROUP BY s.store_id
""")

### SELECT — Subqueries & CTEs

In [ ]:
# Films longer than average
sql("""
    SELECT title, length
    FROM film
    WHERE length > (SELECT AVG(length) FROM film)
    ORDER BY length DESC
    LIMIT 10
""")

In [ ]:
# CTE: most rented films
sql("""
    WITH rental_counts AS (
        SELECT i.film_id, COUNT(*) AS cnt
        FROM rental r
        JOIN inventory i USING (inventory_id)
        GROUP BY i.film_id
    )
    SELECT f.title, rc.cnt AS times_rented
    FROM rental_counts rc
    JOIN film f USING (film_id)
    ORDER BY rc.cnt DESC
    LIMIT 10
""")

---

## Cleanup

Always close the engine when done to free connections, otherwise I might end up with no active connections

In [ ]:
engine.dispose()
print("🔌 Connection pool closed")